# Olist E-Commerce Dimensional Modeling: Silver to Gold Layer
Notebook này đảm nhận vai trò chuyển đổi cấu trúc dữ liệu từ tầng **Silver** lên tầng **Gold**. Dữ liệu tại đây được tổ chức lại theo mô hình **Mô hình thực thể tinh gọn (Star Schema)** gồm các bảng Chiều (Dimensions) và bảng Sự kiện (Facts) nhằm tối ưu hoá hiệu năng truy vấn cho các công cụ OLAP/BI.

### Quản trị kho dữ liệu (Data Governance):
* Sử dụng **Unity Catalog** để quản lý tập trung và phân quyền dữ liệu.
* Cấu hình tối ưu hóa ghi Delta Table (`optimizeWrite`, `autoCompact`) để loại bỏ tình trạng file nhỏ (Small File Problem).

In [0]:
from pyspark.sql import SparkSession
# from pyspark.sql.functions import (
#     col, lower, trim, when, current_timestamp, to_date,
#     year, month, quarter, dayofweek, date_format, lit
# )
import time



# ==== CONFIG ====
storage_account = "mystorageaccount"
catalog_name = "mycatalog"
# catalog_name = "hive_metastore"
silver_schema = "silver"
gold_schema = "gold"

# 1. KẾT NỐI ĐẾN STORAGE ACCOUNT
storage_key = "my-hashkey"
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")


# ==== OPTIMIZE ====
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

start_time = time.time()

# ==== UNITY CATALOG ====
spark.sql(f"USE CATALOG `{catalog_name}`")
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{gold_schema}`")




DataFrame[]

In [0]:

print("Đọc dữ liệu từ Silver...")

df_orders_silver = spark.table(f"{catalog_name}.{silver_schema}.orders")
df_products_silver = spark.table(f"{catalog_name}.{silver_schema}.products")
df_sellers_silver = spark.table(f"{catalog_name}.{silver_schema}.sellers")
df_customers_silver = spark.table(f"{catalog_name}.{silver_schema}.customers")
geolocation = spark.table(f"{catalog_name}.{silver_schema}.geolocation")
df_order_items_silver = spark.table(f"{catalog_name}.{silver_schema}.order_items")
df_order_payments_silver = spark.table(f"{catalog_name}.{silver_schema}.order_payments")
df_order_reviews_silver = spark.table(f"{catalog_name}.{silver_schema}.order_reviews")

display(df_orders_silver.limit(5))


Đọc dữ liệu từ Silver...


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,is_valid,processed_at
f373335aac9a659de916f7170b8bc07a,f06a94a401e52fb019c72f2e8bbf6a2f,shipped,2018-03-17T15:32:31Z,2018-03-17T15:48:40Z,2018-03-20T21:08:28Z,null,2018-04-13T00:00:00Z,null,null,2026-06-16T13:07:52.318162Z
118045506e1c1dda060171af43fe11b4,638c6674418fc58283a73c078bcb076f,delivered,2018-03-08T19:06:05Z,2018-03-09T19:08:26Z,2018-03-13T21:24:28Z,2018-04-11T12:53:50Z,2018-04-04T00:00:00Z,34,true,2026-06-16T13:07:52.318162Z
cc66dee6fbc18bb79903c3a2cc14ff52,19d3b3a2d4756af17603e2c35c7c2815,delivered,2018-04-12T14:37:29Z,2018-04-12T15:15:27Z,2018-04-16T16:23:53Z,2018-04-20T17:28:56Z,2018-05-07T00:00:00Z,8,true,2026-06-16T13:07:52.318162Z
f44cb69655f8e4d13e7aae7cdd3d3c25,eab62436056c6ce3853a17dd6892951a,delivered,2018-07-13T22:22:57Z,2018-07-13T22:35:20Z,2018-07-24T19:07:00Z,2018-07-25T14:03:41Z,2018-07-31T00:00:00Z,12,true,2026-06-16T13:07:52.318162Z
edcc6b79e8394346ba3ba21b00b4055e,08aea10c40f606e52597486db2b56a81,delivered,2018-04-29T16:03:47Z,2018-04-29T16:15:25Z,2018-05-02T08:25:00Z,2018-05-11T23:12:12Z,2018-05-25T00:00:00Z,12,true,2026-06-16T13:07:52.318162Z


## Star Schema Transformation (Thiết kế Dim & Fact)
Thực hiện tách lọc thuộc tính và xây dựng cấu trúc kho dữ liệu:
* **Dimensions (Dim):** `dim_customers`, `dim_sellers`, `dim_products`, và đặc biệt là khởi tạo bảng thời gian động `dim_date` (sinh mã `date_key` dạng chuỗi số nguyên `yyyyMMdd`).
* **Facts:** Gom nhóm tính toán tổng kết (Aggregated Measures) doanh thu, chi phí vận chuyển, điểm đánh giá từ khách hàng để đưa vào `fact_orders`, `fact_order_items`, `fact_order_payments`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

In [0]:
# DIM CUSTOMER
dim_customers = df_customers.select(
    F.col("customer_id"),
    F.col("customer_unique_id"),
    F.col("customer_zip_code_prefix"),
    F.col("customer_city"),
    F.col("customer_state")
).withColumn("gold_updated_at", F.current_timestamp())

# DIM SELLERS
dim_sellers = df_sellers.select(
    F.col("seller_id"),
    F.col("seller_zip_code_prefix"),
    F.col("seller_city"),
    F.col("seller_state")
).withColumn("gold_updated_at", F.current_timestamp())

# DIM PRODUCTS
dim_products = df_products.select(
    F.col("product_id"),
    F.col("product_category_name").alias("category"),
    (F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm")).alias("volume_cm3"),
    F.col("product_weight_g").alias("weight_g")
).withColumn("gold_updated_at", F.current_timestamp())

# DIM DATE
dim_date = df_orders.select(
    F.to_date(F.col("order_purchase_timestamp")).alias("date")
).filter(F.col("date").isNotNull())\
 .distinct()\
 .withColumn("date_key", F.date_format(F.col("date"), "yyyyMMdd").cast(IntegerType()))\
 .withColumn("year", F.year(F.col("date")))\
 .withColumn("month", F.month(F.col("date")))\
 .withColumn("quarter", F.quarter(F.col("date")))\
 .withColumn("day_name", F.date_format(F.col("date"), "EEEE"))\
 .withColumn("month_name", F.date_format(F.col("date"), "MMMM"))\
 .select("date_key", "date", "year", "month", "quarter", "day_name", "month_name")\
 .withColumn("gold_updated_at", F.current_timestamp())

# FACT ORDERS
df_items_agg = df_order_items.groupBy("order_id").agg(
    F.sum("price").alias("total_item_price"),
    F.sum("freight_value").alias("total_freight")
)

df_payments_agg = df_order_payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("total_payment_value")
)

df_reviews_agg = df_order_reviews.groupBy("order_id").agg(
    F.avg("review_score").cast(IntegerType()).alias("review_score")
)

fact_orders = df_orders.alias("o") \
    .join(df_items_agg.alias("i"), "order_id", "left") \
    .join(df_payments_agg.alias("p"), "order_id", "left") \
    .join(df_reviews_agg.alias("r"), "order_id", "left") \
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.date_format(F.col("o.order_purchase_timestamp"), "yyyyMMdd").cast(IntegerType()).alias("order_date_key"),
        F.col("o.order_status"),
        F.col("o.order_purchase_timestamp"),
        F.col("o.order_delivered_customer_date"),
        F.col("o.order_estimated_delivery_date"),
        F.col("o.delivery_days"),
        F.col("o.is_valid_order"),
        F.coalesce(F.col("i.total_item_price"), F.lit(0.0)).alias("total_item_price"),
        F.coalesce(F.col("i.total_freight"), F.lit(0.0)).alias("total_freight"),
        F.coalesce(F.col("p.total_payment_value"), F.lit(0.0)).alias("total_payment_value"),
        F.col("r.review_score")
    ).withColumn("gold_updated_at", F.current_timestamp())

# FACT ORDER_ITEMS
fact_order_items = df_order_items.alias("oi") \
    .join(df_orders.alias("o"), "order_id", "left") \
    .select(
        F.col("oi.order_id"),
        F.col("oi.order_item_id").alias("fact_item_id"),
        F.col("oi.product_id"),
        F.col("oi.seller_id"),
        F.date_format(F.col("o.order_purchase_timestamp"), "yyyyMMdd").cast(IntegerType()).alias("order_date_key"),
        F.col("oi.price"),
        F.col("oi.freight_value")
    ).withColumn("gold_updated_at", F.current_timestamp())

#FACT ORDER_PAYMENT
fact_order_payments = df_order_payments.alias("op") \
    .join(df_orders.alias("o"), "order_id", "left") \
    .select(
        F.col("op.order_id"),
        F.date_format(F.col("o.order_purchase_timestamp"), "yyyyMMdd").cast(IntegerType()).alias("order_date_key"),
        F.col("op.payment_sequential").alias("fact_payment_id"),
        F.col("op.payment_type"),
        F.col("op.payment_installments"),
        F.col("op.payment_value")
    ).withColumn("gold_updated_at", F.current_timestamp())


✅ Đã tạo xong các bảng Gold


## Compute RFM Metrics (Tính toán chỉ số phân cụm khách hàng)

Đoạn code này thực hiện trích xuất bộ 3 chỉ số cốt lõi của mô hình **RFM (Recency, Frequency, Monetary)** dựa trên điểm mốc thời gian cuối cùng của hệ thống (`max_system_date`). Dữ liệu được nhóm (Group by) theo từng khách hàng duy nhất để phục vụ cho bài toán phân cụm chiến lược trên Power BI.

### Ý nghĩa hệ thống chỉ số:
* **Recency (R) - Độ tươi mới:** Tính khoảng cách (số ngày) từ ngày mua hàng cuối cùng của khách đến điểm neo hệ thống. Số ngày càng nhỏ chứng tỏ khách hàng vừa mới mua sắm gần đây.
* **Frequency (F) - Tần suất:** Đếm tổng số lượng đơn hàng duy nhất (`order_id`) mà khách đã thực hiện. Giúp xác định mức độ thường xuyên quay lại của tệp khách.
* **Monetary (M) - Giá trị tiền tệ:** Tổng hợp toàn bộ số tiền thanh toán (`total_payment_value`) mà khách hàng đã đóng góp cho sàn thương mại điện tử.

In [ ]:
# FACT RFM:
max_system_date = fact_orders.agg(F.max("order_purchase_timestamp")).collect()[0][0]

fact_crm_rfm = fact_orders.alias("o") \
    .join(dim_customers.alias("c"), "customer_id", "inner") \
    .groupBy("c.customer_unique_id", "c.customer_state") \
    .agg(
        # Recency: Số ngày từ lần mua cuối của khách hàng đến ngày cuối cùng của hệ thống
        F.datediff(F.lit(max_system_date), F.max("o.order_purchase_timestamp")).alias("recency"),
        # Frequency: Tổng số đơn hàng duy nhất của khách hàng đó
        F.countDistinct ("o.order_id").alias("frequency"),
        # Monetary: Tổng số tiền thanh toán của khách hàng đó
        F.sum("o.total_payment_value").alias("monetary_")
    )

## Unity Catalog Persistence & Delta Optimization
Định nghĩa quy trình lưu trữ tự động hóa: Ghi toàn bộ các thực thể Dim/Fact vào schema hệ thống của Unity Catalog, đồng thời in thông báo xác nhận trạng thái hoàn thành của toàn bộ Layer.

In [0]:

def save_uc_table(df, table_name):
    full_name = f"{catalog_name}.{gold_schema}.{table_name}"
    
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name))
    
    print(f"✅ Saved: {full_name}")

# ==== WRITE ALL GOLD TABLES ====

gold_tables = {
    "fact_orders": fact_orders,
    "fact_order_items": fact_order_items,
    "fact_order_payments":fact_order_payments,
    "fact_rfm": fact_crm_rfm,
    "dim_customers": dim_customers,
    "dim_sellers": dim_sellers,
    "dim_products": dim_products,
    "dim_date": dim_date,
}

for name, df in gold_tables.items():
    save_uc_table(df, name)


print("✅ Hoàn thành Gold layer trong Unity Catalog.")